<div style="display:flex; align-items:center; justify-content:space-between; gap:24px; border-bottom:1px solid #d0d7de; padding-bottom:12px; margin-bottom:18px;">
  <div>
    <h1 style="margin:0;">Infraestructuras Computacionales para el Procesamiento de datos masivos</h1>
    <p style="margin:6px 0 0 0;"><strong>Ejercicio propuesto, Modulo 3 Tema 3 - Kafka + Spark Structured Streaming avanzado</strong></p>
  </div>
  <img src="https://www.uned.es/universidad/.imaging/mte/site-facultades-theme/220/dam/recursos-corporativos/logotipos/facultades-escuelas/logo_informatica.gif/jcr:content/logo_informatica.gif" alt="Logo ETSI Informatica UNED" style="height:72px; width:auto;" />
</div>

## Kafka + Spark Structured Streaming avanzado

Este cuaderno transforma la pagina `M3T3L5.html` en una practica ejecutable paso a paso. El objetivo es construir una evolucion avanzada del pipeline de streaming del tema anterior: Spark Structured Streaming consume eventos reales desde Kafka, valida mensajes JSON, separa datos correctos y erroneos, calcula metricas, genera alertas y publica eventos enriquecidos en un segundo topic Kafka.

### Objetivos

1. Levantar Kafka en modo KRaft y un contenedor de laboratorio con PySpark.
2. Crear los topics `orders` y `orders_enriched`.
3. Generar eventos JSON de pedidos con un productor Python.
4. Consumir Kafka desde Spark Structured Streaming con `spark-sql-kafka-0-10`.
5. Validar eventos y separar datos validos de datos en cuarentena.
6. Calcular latencia extremo a extremo y metricas por comercio, pais y canal.
7. Detectar alertas por importes altos o eventos muy tardios.
8. Persistir resultados en Parquet y publicar eventos enriquecidos en Kafka.


## 1. Arquitectura del laboratorio

El flujo de datos de la practica es:

```text
producer_orders.py
        |
        v
Kafka topic: orders
        |
        v
Spark Structured Streaming
        |-- output/silver_orders
        |-- output/quarantine_orders
        |-- output/merchant_metrics
        |-- output/alerts
        |
        v
Kafka topic: orders_enriched
```

El productor genera pedidos y transacciones de comercio electronico. Algunos eventos son validos, otros llegan tarde y otros son erroneos para comprobar la validacion y la cuarentena.


## 2. Ficheros disponibles

La carpeta `kafka_spark_advanced_lab/` contiene estos ficheros principales:

- `Dockerfile`: imagen del contenedor de laboratorio con Python, Java y PySpark.
- `docker-compose.yml`: servicios `kafka` y `lab`.
- `requirements.txt`: dependencias Python.
- `app/producer_orders.py`: productor Kafka de pedidos.
- `app/streaming_pipeline.py`: pipeline Spark Structured Streaming.
- `app/validate_outputs.py`: lectura batch de resultados Parquet.
- `README.md`: guia rapida de ejecucion.

Este cuaderno debe ejecutarse desde la carpeta `modulo3/Tema3/kafka_spark_advanced_lab` para que los comandos Docker encuentren el `docker-compose.yml`.


In [1]:
from pathlib import Path

base_dir = Path.cwd()
lab_dir = base_dir / "kafka_spark_advanced_lab"

if lab_dir.exists():
    %cd kafka_spark_advanced_lab
    base_dir = Path.cwd()

print("Directorio actual:", base_dir)

expected = [
    "Dockerfile",
    "docker-compose.yml",
    "requirements.txt",
    "README.md",
    "app/producer_orders.py",
    "app/streaming_pipeline.py",
    "app/validate_outputs.py",
]

for name in expected:
    path = base_dir / name
    print(f"{name:36} ->", "OK" if path.exists() else "NO EXISTE")


/Users/noelia/PycharmProjects/PFG/modulo3/Tema3/kafka_spark_advanced_lab
Directorio actual: /Users/noelia/PycharmProjects/PFG/modulo3/Tema3/kafka_spark_advanced_lab
Dockerfile                           -> OK
docker-compose.yml                   -> OK
requirements.txt                     -> OK
README.md                            -> OK
app/producer_orders.py               -> OK
app/streaming_pipeline.py            -> OK
app/validate_outputs.py              -> OK


/Users/noelia/PycharmProjects/PFG/.venv/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


## 3. Requisitos

Necesitas Docker y Docker Compose. En macOS y Windows se puede usar Docker Desktop. En Linux, Docker Engine con el plugin de Compose.

El primer `spark-submit --packages` necesita conexion a internet para descargar el conector Spark-Kafka desde Maven.


In [2]:
!docker --version
!docker compose version
!docker run --rm hello-world


Docker version 29.5.3, build d1c06ef
Docker Compose version v5.1.4

Hello from Docker!
This message shows that your installation appears to be working correctly.

To generate this message, Docker took the following steps:
 1. The Docker client contacted the Docker daemon.
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
    (arm64v8)
 3. The Docker daemon created a new container from that image which runs the
    executable that produces the output you are currently reading.
 4. The Docker daemon streamed that output to the Docker client, which sent it
    to your terminal.

To try something more ambitious, you can run an Ubuntu container with:
 $ docker run -it ubuntu bash

Share images, automate workflows, and more with a free Docker ID:
 https://hub.docker.com/

For more examples and ideas, visit:
 https://docs.docker.com/get-started/



## 4. Construir la imagen del laboratorio

La primera construccion puede tardar varios minutos porque descarga dependencias del sistema y paquetes Python.

Resultado esperado: construccion sin errores y mensaje final similar a `DONE` o `Successfully built`.


In [3]:
!docker compose build


[+] Building 0.0s (0/1)                                                         
 => [internal] load local bake definitions                                 0.0s
[+] Building 0.1s (1/2)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 618B                                             0.0s
[+] Building 0.3s (2/3)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 618B                                             0.0s
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 459B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim-bookw  0.2s
[+] Building 0.4s (2/3)                                                         
 => [internal] load local bake defin

Si hay un error de paquetes durante la construccion, reconstruye la imagen del servicio `lab` sin cache.


In [ ]:
# Ejecutar solo si falla la construccion anterior.
# !docker compose build --no-cache lab


## 5. Arrancar Kafka

Arranca el broker Kafka y comprueba que el contenedor queda activo. El laboratorio usa Kafka en modo KRaft, por lo que no necesita ZooKeeper.


In [14]:
!docker compose up -d kafka
!docker compose ps


[+] up 1/1
 ✔ Container m3t3-advanced-kafka Running                                    0.0s

What's next:
    Filter, search, and stream logs from all your Compose services
    in one place with Docker Desktop's Logs view. ]8;;docker-desktop://dashboard/logs?appId=kafka_spark_advanced_lab\docker-desktop://dashboard/logs?appId=kafka_spark_advanced_lab]8;;\
NAME                  IMAGE                COMMAND                  SERVICE   CREATED          STATUS                    PORTS
m3t3-advanced-kafka   apache/kafka:4.2.0   "/__cacert_entrypoin…"   kafka     39 seconds ago   Up 38 seconds (healthy)   0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp


## 6. Crear los topics Kafka

El topic `orders` recibira los eventos JSON originales. El topic `orders_enriched` recibira los eventos validos despues de ser procesados y enriquecidos por Spark.


In [15]:
!docker compose exec kafka /opt/kafka/bin/kafka-topics.sh --bootstrap-server kafka:9092 --create --if-not-exists --topic orders --partitions 3 --replication-factor 1
!docker compose exec kafka /opt/kafka/bin/kafka-topics.sh --bootstrap-server kafka:9092 --create --if-not-exists --topic orders_enriched --partitions 3 --replication-factor 1


Created topic orders.
Created topic orders_enriched.


Verifica la configuracion del topic de entrada. Deben aparecer tres particiones y factor de replicacion 1, adecuado para un laboratorio local con un solo broker.


In [16]:
!docker compose exec kafka /opt/kafka/bin/kafka-topics.sh --bootstrap-server kafka:9092 --describe --topic orders


Topic: orders	TopicId: aN-8qJU3T8WUxbb-WyajOQ	PartitionCount: 3	ReplicationFactor: 1	Configs: min.insync.replicas=1
	Topic: orders	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
	Topic: orders	Partition: 1	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
	Topic: orders	Partition: 2	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 


## 7. Producir eventos JSON

El productor genera eventos de pedidos. Con `--invalid-rate 0.03` se fuerza aproximadamente un 3% de eventos erroneos; con `--late-rate 0.08` se fuerza aproximadamente un 8% de eventos tardios.

Salida esperada:

```text
Eventos enviados: 100
Eventos enviados: 200
...
Finalizado. Total eventos enviados: 1000
```


In [17]:
!docker compose run --rm lab python /workspace/app/producer_orders.py --bootstrap-server kafka:9092 --topic orders --messages 1000 --sleep 0.01 --invalid-rate 0.03 --late-rate 0.08


[+] create 1/1
 ✔ Container m3t3-advanced-kafka Running                                    0.0s
[+] start 1/1
 ✔ Container m3t3-advanced-kafka Running                                    0.0s
[+]  1/1
 ✔ Container m3t3-advanced-kafka Running                                    0.0s
Container m3t3-advanced-kafka Waiting 
Container m3t3-advanced-kafka Healthy 
Container kafka_spark_advanced_lab-lab-run-0450d23b3674 Creating 
Container kafka_spark_advanced_lab-lab-run-0450d23b3674 Created 
Eventos enviados: 100
Eventos enviados: 200
Eventos enviados: 300
Eventos enviados: 400
Eventos enviados: 500
Eventos enviados: 600
Eventos enviados: 700
Eventos enviados: 800
Eventos enviados: 900
Eventos enviados: 1000
Finalizado. Total eventos enviados: 1000


## 8. Inspeccionar mensajes en Kafka

Consume algunos mensajes desde el principio del topic para comprobar que se han publicado correctamente. Deben verse mensajes JSON con campos como `event_id`, `event_time`, `customer_id`, `merchant_id`, `country`, `channel` y `amount`.


In [18]:
!docker compose exec kafka /opt/kafka/bin/kafka-console-consumer.sh --bootstrap-server kafka:9092 --topic orders --from-beginning --max-messages 5


{"event_id": "9a4d2b40-86ec-499a-a834-d59c8ff752f7", "event_time": "2026-06-22T17:48:26Z", "source": "orders-simulator", "customer_id": "c-0040", "merchant_id": "m-400", "country": "IT", "channel": "pos", "amount": 65.12, "currency": "EUR", "device_id": "d-0084"}
{"event_id": "27bc00f5-6e55-4f73-9f98-7ba2e3dea7f7", "event_time": "2026-06-22T17:48:26Z", "source": "orders-simulator", "customer_id": "c-0045", "merchant_id": "m-200", "country": "IT", "channel": "web", "amount": 69.15, "currency": "EUR", "device_id": "d-0124"}
{"event_id": "3a7ce895-e2bd-4c75-9a3e-b7a430324998", "event_time": "2026-06-22T17:48:26Z", "source": "orders-simulator", "customer_id": "c-0020", "merchant_id": "m-300", "country": "NL", "channel": "pos", "amount": 55.08, "currency": "EUR", "device_id": "d-0077"}
{"event_id": "dba8f68c-bfe4-4605-a33f-77869cbde862", "event_time": "2026-06-22T17:48:26Z", "source": "orders-simulator", "customer_id": "c-0054", "merchant_id": "m-400", "country": "FR", "channel": "pos", "am

## 9. Ejecutar Spark Structured Streaming

Spark consume `orders`, parsea el JSON con esquema explicito, valida campos obligatorios, calcula latencia, separa cuarentena, genera metricas por comercio/pais/canal, crea alertas y publica eventos validos enriquecidos en `orders_enriched`.

La consulta usa `trigger(availableNow=True)`, por lo que procesa los datos disponibles y finaliza.

Salida esperada:

```text
Batch 0: validos=..., invalidos=..., metricas=..., alertas=...
Consulta finalizada.
```


In [21]:
!docker compose run --rm lab spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1 /workspace/app/streaming_pipeline.py --bootstrap-server kafka:9092 --input-topic orders --enriched-topic orders_enriched --output /workspace/output --checkpoint /workspace/checkpoint/orders_stream


[+] create 1/1
 ✔ Container m3t3-advanced-kafka Running                                    0.0s
[+] start 1/1
 ✔ Container m3t3-advanced-kafka Running                                    0.0s
[+]  1/1
 ✔ Container m3t3-advanced-kafka Running                                    0.0s
Container m3t3-advanced-kafka Waiting 
Container m3t3-advanced-kafka Healthy 
Container kafka_spark_advanced_lab-lab-run-eb62e086b6ee Creating 
Container kafka_spark_advanced_lab-lab-run-eb62e086b6ee Created 
:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bad795cd-9727-4ec0-8c41-83160eed4be4;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	fou

## 10. Comprobar el topic enriquecido

El job Spark escribe de nuevo en Kafka solo los eventos validos, ya enriquecidos con campos como `amount_bucket`, `latency_seconds` y `batch_id`.


In [22]:
!docker compose exec kafka /opt/kafka/bin/kafka-console-consumer.sh --bootstrap-server kafka:9092 --topic orders_enriched --from-beginning --max-messages 5


{"kafka_key":"00503320-bb3c-44f0-93fe-52fc0a8bf2f8","raw_value":"{\"event_id\": \"00503320-bb3c-44f0-93fe-52fc0a8bf2f8\", \"event_time\": \"2026-06-22T17:48:35Z\", \"source\": \"orders-simulator\", \"customer_id\": \"c-0187\", \"merchant_id\": \"m-100\", \"country\": \"FR\", \"channel\": \"api\", \"amount\": 38.48, \"currency\": \"EUR\", \"device_id\": \"d-0080\"}","topic":"orders","partition":0,"offset":244,"kafka_timestamp":"2026-06-22T17:48:35.601Z","event_id":"00503320-bb3c-44f0-93fe-52fc0a8bf2f8","event_time":"2026-06-22T17:48:35Z","source":"orders-simulator","customer_id":"c-0187","merchant_id":"m-100","country":"FR","channel":"api","amount":38.48,"currency":"EUR","device_id":"d-0080","event_ts":"2026-06-22T17:48:35.000Z","processed_time":"2026-06-22T17:51:39.169Z","event_window_5m":{"start":"2026-06-22T17:45:00.000Z","end":"2026-06-22T17:50:00.000Z"},"event_date":"2026-06-22","amount_bucket":"normal","latency_seconds":184,"batch_id":0}
{"kafka_key":"000f5f5d-8163-41a4-a803-d6070

## 11. Validar resultados Parquet

El validador lee las salidas Parquet generadas por Spark y muestra conteos, esquemas y muestras de datos.

Resultado esperado:

```text
== silver_orders ==
Registros: ...

== quarantine_orders ==
Registros: ...

== merchant_metrics ==
Registros: ...

== alerts ==
Registros: ...
```

La cuarentena puede contener registros si se uso `--invalid-rate` mayor que cero. `alerts` puede contener operaciones de importe alto o eventos con latencia superior a 600 segundos.


In [23]:
!docker compose run --rm lab python /workspace/app/validate_outputs.py


[+] create 1/1
 ✔ Container m3t3-advanced-kafka Running                                    0.0s
[+] start 1/1
 ✔ Container m3t3-advanced-kafka Running                                    0.0s
[+]  1/1
 ✔ Container m3t3-advanced-kafka Running                                    0.0s
Container m3t3-advanced-kafka Waiting 
Container m3t3-advanced-kafka Healthy 
Container kafka_spark_advanced_lab-lab-run-3e9cc7d58ab6 Creating 
Container kafka_spark_advanced_lab-lab-run-3e9cc7d58ab6 Created 
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/22 17:52:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable

== silver_orders ==
Registros: 975
root
 |-- kafka_key: string (nullable = true)
 |-- raw_value: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)

Tambien puedes revisar los ficheros generados desde la maquina anfitriona.


In [25]:
from pathlib import Path

for folder in [Path("output"), Path("checkpoint")]:
    print(f"\n== {folder} ==")
    if not folder.exists():
        print("No existe")
        continue
    files = sorted(p for p in folder.rglob("*") if p.is_file())
    if not files:
        print("Sin ficheros")
        continue
    for file in files[:80]:
        print(file)
    if len(files) > 80:
        print(f"... {len(files) - 80} ficheros mas")



== output ==
output/alerts/._SUCCESS.crc
output/alerts/.part-00000-1a9dd532-09ad-4552-87ca-d30b152f9f2b-c000.snappy.parquet.crc
output/alerts/.part-00001-1a9dd532-09ad-4552-87ca-d30b152f9f2b-c000.snappy.parquet.crc
output/alerts/.part-00002-1a9dd532-09ad-4552-87ca-d30b152f9f2b-c000.snappy.parquet.crc
output/alerts/.part-00003-1a9dd532-09ad-4552-87ca-d30b152f9f2b-c000.snappy.parquet.crc
output/alerts/_SUCCESS
output/alerts/part-00000-1a9dd532-09ad-4552-87ca-d30b152f9f2b-c000.snappy.parquet
output/alerts/part-00001-1a9dd532-09ad-4552-87ca-d30b152f9f2b-c000.snappy.parquet
output/alerts/part-00002-1a9dd532-09ad-4552-87ca-d30b152f9f2b-c000.snappy.parquet
output/alerts/part-00003-1a9dd532-09ad-4552-87ca-d30b152f9f2b-c000.snappy.parquet
output/merchant_metrics/._SUCCESS.crc
output/merchant_metrics/.part-00000-17aa329c-248a-4735-9c05-8cefb480f304-c000.snappy.parquet.crc
output/merchant_metrics/.part-00001-17aa329c-248a-4735-9c05-8cefb480f304-c000.snappy.parquet.crc
output/merchant_metrics/.pa

## 12. Repetir una prueba desde cero

Para limpiar resultados y checkpoints de una ejecucion anterior, borra `output/` y `checkpoint/`. Si quieres borrar tambien los datos internos de Kafka, usa `docker compose down -v`.


In [ ]:
# Ejecutar solo si quieres reiniciar completamente la prueba.
# En Linux/macOS/WSL:
# !rm -rf output/* checkpoint/*
# !docker compose down -v


## 13. Parar el laboratorio

Para parar los contenedores sin borrar volumenes internos:


In [26]:
!docker compose down


[+] down 0/1
 ⠋ Container m3t3-advanced-kafka Stopping                                   0.1s
[+] down 0/1
 ⠙ Container m3t3-advanced-kafka Stopping                                   0.2s
[+] down 0/1
 ⠹ Container m3t3-advanced-kafka Stopping                                   0.3s
[+] down 0/1
 ⠸ Container m3t3-advanced-kafka Stopping                                   0.4s
[+] down 0/1
 ⠼ Container m3t3-advanced-kafka Stopping                                   0.5s
[+] down 0/1
 ⠴ Container m3t3-advanced-kafka Stopping                                   0.6s
[+] down 0/1
 ⠦ Container m3t3-advanced-kafka Stopping                                   0.7s
[+] down 0/1
 ⠧ Container m3t3-advanced-kafka Stopping                                   0.8s
[+] down 0/1
 ⠇ Container m3t3-advanced-kafka Stopping                                   0.9s
[+] down 1/2
 ✔ Container m3t3-advanced-kafka            Removed                         1.0s
 ⠋ Network kafka_spark_advanced_lab_default Removing        

## 14. Ampliaciones avanzadas con respuestas

1. **Agregacion por ventanas:** anade una agregacion por ventanas de 5 minutos con watermark de 10 minutos.

<details>
<summary><strong>Ver solucion del ejercicio 1</strong></summary>

En `streaming_pipeline.py` ya se calcula `event_window_5m` con `window(col("event_ts"), "5 minutes")`. Para convertirlo en una agregacion streaming con control de datos tardios, se puede crear otro flujo antes del `foreachBatch`:

```python
window_metrics = (
    parsed
    .withWatermark("event_ts", "10 minutes")
    .filter(col("event_id").isNotNull())
    .filter(col("amount").isNotNull() & (col("amount") > 0))
    .groupBy(window("event_ts", "5 minutes"), "country", "channel")
    .agg(
        count("*").alias("events"),
        spark_sum("amount").alias("total_amount"),
        avg("amount").alias("avg_amount")
    )
)
```

El watermark permite a Spark mantener estado acotado y cerrar ventanas cuando ya no espera eventos mas antiguos que 10 minutos.

</details>

2. **Usuarios unicos aproximados:** calcula usuarios unicos aproximados por pais y canal con `approx_count_distinct`.

<details>
<summary><strong>Ver solucion del ejercicio 2</strong></summary>

Importa la funcion:

```python
from pyspark.sql.functions import approx_count_distinct
```

Despues anade la metrica en `merchant_metrics` o crea una nueva salida agregada:

```python
country_channel_metrics = (
    valid
    .groupBy("country", "channel")
    .agg(
        count("*").alias("events"),
        approx_count_distinct("customer_id").alias("approx_unique_customers"),
        spark_sum("amount").alias("total_amount")
    )
    .withColumn("batch_id", lit(batch_id))
)
```

`approx_count_distinct` reduce el coste frente a un conteo exacto cuando el volumen de datos es alto.

</details>

3. **Regla de fraude:** implementa una regla por cliente: mas de 5 operaciones en 2 minutos o importe acumulado superior a 1500 EUR.

<details>
<summary><strong>Ver solucion del ejercicio 3</strong></summary>

Una forma didactica es crear una agregacion por ventana y cliente:

```python
fraud_candidates = (
    parsed
    .withWatermark("event_ts", "10 minutes")
    .filter(col("customer_id").isNotNull())
    .filter(col("amount").isNotNull() & (col("amount") > 0))
    .groupBy(window("event_ts", "2 minutes"), "customer_id")
    .agg(
        count("*").alias("operations"),
        spark_sum("amount").alias("total_amount")
    )
    .filter((col("operations") > 5) | (col("total_amount") > 1500))
)
```

En produccion se deberia enriquecer con historico del cliente, reglas parametrizables y un sink idempotente para alertas.

</details>

4. **Clave Kafka por cliente:** cambia el productor para generar claves Kafka por `customer_id` y analiza la distribucion por particiones.

<details>
<summary><strong>Ver solucion del ejercicio 4</strong></summary>

En `producer_orders.py`, sustituye la clave usada para eventos validos:

```python
event = build_order(args.source, args.late_rate)
key = event["customer_id"]
payload = json.dumps(event)
```

Con esta clave, Kafka envia todos los eventos de un mismo cliente a la misma particion, siempre que el numero de particiones no cambie. Esto facilita procesamiento ordenado por cliente, pero puede generar desbalanceo si algunos clientes concentran mucho trafico.

Para analizar particiones, revisa las columnas `partition` y `offset` que Spark conserva desde Kafka.

</details>

5. **Compresion en el productor:** activa compresion y compara el tiempo de envio.

<details>
<summary><strong>Ver solucion del ejercicio 5</strong></summary>

En la creacion del `KafkaProducer`, anade `compression_type`. Por ejemplo:

```python
producer = KafkaProducer(
    bootstrap_servers=args.bootstrap_server,
    key_serializer=lambda value: value.encode("utf-8"),
    value_serializer=lambda value: value.encode("utf-8"),
    acks="all",
    retries=5,
    compression_type="gzip",
)
```

Despues ejecuta dos pruebas con el mismo numero de mensajes y compara el tiempo. La compresion reduce bytes enviados, pero consume CPU adicional.

</details>

6. **Checkpoint e idempotencia:** ejecuta el job dos veces con el mismo checkpoint y explica por que no reprocesa offsets.

<details>
<summary><strong>Ver solucion del ejercicio 6</strong></summary>

Spark Structured Streaming guarda en el checkpoint los offsets ya confirmados. Si ejecutas el mismo comando con el mismo `--checkpoint` y no hay mensajes nuevos, Spark continua desde el ultimo offset procesado y no vuelve a leer los mensajes anteriores.

Esto protege frente a reprocesamiento accidental, pero no convierte automaticamente todos los sinks en idempotentes. Si un batch falla despues de escribir en Parquet y antes de confirmar offsets, pueden aparecer duplicados si no se deduplica por `event_id` o si el sink no soporta transacciones.

</details>

7. **Borrar solo el checkpoint:** conserva Kafka, borra el checkpoint, vuelve a ejecutar y documenta el reprocesamiento.

<details>
<summary><strong>Ver solucion del ejercicio 7</strong></summary>

Si borras `checkpoint/orders_stream` pero mantienes Kafka, la consulta pierde la memoria de offsets. Como el script usa `startingOffsets=earliest`, Spark vuelve a leer desde el principio del topic.

Comandos de prueba:

```bash
rm -rf checkpoint/orders_stream
docker compose run --rm lab spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1 /workspace/app/streaming_pipeline.py --bootstrap-server kafka:9092 --input-topic orders --enriched-topic orders_enriched --output /workspace/output --checkpoint /workspace/checkpoint/orders_stream
```

Si no limpias `output/`, pueden duplicarse resultados en Parquet. Para una repeticion limpia, borra tambien `output/*`.

</details>

8. **Topic de alertas:** publica alertas en un tercer topic `orders_alerts`.

<details>
<summary><strong>Ver solucion del ejercicio 8</strong></summary>

Primero crea el topic:

```bash
docker compose exec kafka /opt/kafka/bin/kafka-topics.sh --bootstrap-server kafka:9092 --create --if-not-exists --topic orders_alerts --partitions 3 --replication-factor 1
```

Dentro de `process_batch`, prepara `key` y `value` para Kafka:

```python
alerts_for_kafka = (
    alerts
    .withColumn("value", to_json(struct(*[col(name) for name in alerts.columns])))
    .selectExpr("CAST(event_id AS STRING) AS key", "CAST(value AS STRING) AS value")
)

alerts_for_kafka.write.format("kafka") \
    .option("kafka.bootstrap.servers", bootstrap_server) \
    .option("topic", "orders_alerts") \
    .save()
```

Al ser escritura batch dentro de `foreachBatch`, la publicacion se realiza una vez por microbatch.

</details>

9. **Sustituir Parquet por Delta Lake:** razona que mejora en actualizaciones e idempotencia.

<details>
<summary><strong>Ver solucion del ejercicio 9</strong></summary>

Delta Lake aporta transacciones ACID sobre almacenamiento de ficheros, control de versiones, `MERGE`, actualizaciones y borrados. Para este laboratorio permitiria escribir eventos usando `event_id` como clave logica y reducir duplicados mediante operaciones de upsert.

Con Parquet simple, Spark solo anade ficheros. Si se reprocesa un batch, es facil duplicar registros. Con Delta, se puede implementar una estrategia mas robusta:

```python
# Idea conceptual: usar MERGE sobre event_id en una tabla Delta.
# Requiere anadir dependencias delta-spark y configurar SparkSession.
```

La mejora principal es que el sink deja de ser solo append y puede gestionar reintentos, deduplicacion y correcciones de datos.

</details>

10. **Produccion frente a laboratorio:** documenta que decisiones son de laboratorio y cuales serian necesarias en produccion.

<details>
<summary><strong>Ver solucion del ejercicio 10</strong></summary>

| Aspecto | Laboratorio | Produccion |
|---|---|---|
| Kafka | Un broker local, factor de replicacion 1 | Cluster con varios brokers, replicacion y politicas de retencion |
| Seguridad | Sin autenticacion ni cifrado | TLS, SASL, ACLs y gestion de secretos |
| Spark | `local[*]` dentro de un contenedor | Cluster Spark, Kubernetes, YARN o plataforma gestionada |
| Sinks | Parquet append | Sinks transaccionales o idempotentes, particionado y compactacion |
| Observabilidad | Logs y validacion manual | Metricas, alertas, trazas, dashboards y SLAs |
| Errores | Cuarentena local | Dead-letter topics, reintentos, catalogo de errores y reprocesamiento controlado |
| Esquemas | JSON con esquema en codigo | Schema Registry, versionado y compatibilidad de contratos |

La practica mantiene la arquitectura conceptual real, pero reduce infraestructura y seguridad para que el estudiante pueda ejecutarla en local.

</details>


## 15. Problemas frecuentes

### Kafka no esta listo

Espera unos segundos y revisa el estado:

```bash
docker compose ps
docker compose logs kafka --tail=50
```

### `UnknownHostException: kafka`

Este error suele aparecer cuando se ejecuta Spark fuera de Docker. La guia esta pensada para ejecutar productor y Spark dentro del servicio `lab`, usando `kafka:9092` como bootstrap server.

### Error descargando paquetes Maven

El primer `spark-submit --packages` necesita internet para descargar `org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1`. Repite cuando haya red o configura un repositorio Maven interno.

### No aparecen nuevos datos

Si usas el mismo checkpoint, Spark continua desde el ultimo offset procesado. Para una prueba desde cero:

```bash
rm -rf checkpoint/* output/*
```

### La cuarentena aparece vacia

Puede ocurrir si la ejecucion genero pocos mensajes invalidos. Aumenta `--messages` o `--invalid-rate`, por ejemplo `--invalid-rate 0.10`.

### Hay muchos ficheros pequenos

Es normal en laboratorios de streaming con microbatches. En produccion se ajustan particiones, frecuencia de trigger y procesos de compactacion.


## 16. Resultado esperado

Tras ejecutar los pasos principales, el laboratorio debe demostrar:

- mensajes JSON publicados en `orders`;
- registros validos enriquecidos en `silver_orders`;
- registros erroneos en `quarantine_orders`, si se genero algun evento invalido;
- metricas agregadas en `merchant_metrics`;
- alertas por importe alto o evento tardio en `alerts`;
- mensajes enriquecidos publicados en `orders_enriched`.

El punto clave del ejercicio es comprobar el comportamiento extremo a extremo: productor, Kafka, Structured Streaming, checkpoints, sinks Parquet y topic de salida.
